# Google Cloud Vertex AI — Deploying a Model Endpoint

Vertex AI is Google Cloud's managed ML platform. Like SageMaker and Azure ML, it handles the infrastructure for hosting and serving models. You upload your trained model, create an endpoint, deploy the model to it, and call the endpoint's prediction API.

This notebook covers the full deployment flow. Cloud SDK calls are wrapped in `try/except` to handle missing credentials gracefully — you can read and run every cell.

## 🔗 Where this fits

**Builds on:** Course 05 (AIAT 115) — Unit 5, lesson 08 "08. Deployment & Monitoring" — the same goal as the local API you built there, with the endpoint, autoscaling and traffic splitting supplied by the platform rather than by your laptop.


## 📰 12 June 2025: a blank field took Google Cloud down worldwide

Google added a new quota-policy feature to **Service Control**, the component that authorises API requests across Google Cloud. The new code path had no error handling and, critically, **no feature-flag protection**. On **12 June 2025** a policy change containing unintended blank fields was pushed to the regional Spanner database that Service Control reads. The blank fields triggered a null-pointer exception; Service Control binaries began crash-looping — **globally, in every region at once**, because the policy data replicates worldwide within seconds. More than 50 Google Cloud products were affected for roughly **three hours** (10:49–13:49 PDT), and recovery in `us-central1` took longer still because the restarting tasks lacked randomized exponential backoff and overwhelmed the infrastructure they depended on.

Google's own analysis contains the line every deployment engineer should copy into their notes: the release *"went through our region by region rollout, but the code path that failed was never exercised"* — because the data that would trigger it did not exist until later. A staged rollout protects you from bad **code**. It does not protect you from bad **data** arriving after the code is everywhere.

**What goes wrong without this lesson.** Vertex AI, like SageMaker and Azure ML, hands you an HTTPS endpoint and takes the infrastructure off your plate. What it does not take away is the consequence of that infrastructure failing, or the discipline of knowing which model is behind the URL and how to move traffic off it. Understanding Model versus Endpoint versus deployed model is what makes those actions possible in an incident.

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain the difference between a Vertex AI Model object and an Endpoint object
2. Understand why Vertex AI requires the model artifact to be on GCS first
3. Read the `aiplatform.Model.upload()`, `Endpoint.create()`, and `model.deploy()` calls
4. Interpret a prediction response from a Vertex AI endpoint
5. Describe at least one difference between Vertex AI and SageMaker

## 1. Vertex AI Concepts

**Model** — a registered artifact. Points to a GCS path where your model files live, and to a container image that knows how to load and serve them. (Container image = the pre-packaged runtime box introduced in notebook 01's primer; Unit 4 teaches how to build your own.)

**Endpoint** — a stable HTTPS resource that receives prediction requests. A model must be *deployed* to an endpoint before it can receive traffic.

**Deployment** — the act of allocating compute to serve a model through an endpoint. You can deploy multiple model versions to one endpoint and split traffic between them.

**GCS (Google Cloud Storage)** — Google's object storage, equivalent to AWS S3. Vertex AI pulls the model artifact from GCS.

```
GCS Bucket
  gs://my-bucket/models/iris/v1/model.joblib
        |
        v
Vertex AI Model  <-- registered model pointing to GCS path
        |
        v (deploy)
Vertex AI Endpoint  <-- HTTPS URL for predictions
        |
        v
Your application calls endpoint.predict(instances=[...])
```

## 2. Install Dependencies

In [1]:
# WHAT: install the Vertex AI SDK and the Cloud Storage client.
# WHY: the SDK works offline for building request objects — students without a
# GCP project can still follow every step of the deployment flow.
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'google-cloud-aiplatform', 'google-cloud-storage', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('google-cloud-aiplatform and google-cloud-storage installed')
else:
    print(result.stdout[-500:] if result.stdout else result.stderr[-500:])

google-cloud-aiplatform and google-cloud-storage installed


## 3. Train and Save the Model

In [2]:
# WHAT: train the iris model and stage model.joblib + scaler.joblib in one folder.
# WHY: Vertex AI's pre-built sklearn container expects a model artifact folder in
# GCS — this local directory is what we will upload next.
import joblib
import pathlib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_s, y_train)

print(f"Test accuracy: {clf.score(scaler.transform(X_test), y_test):.2%}")

# One folder = one model version; mirrors the SageMaker/Azure staging pattern.
model_dir = pathlib.Path('/tmp/vertex_model')
model_dir.mkdir(exist_ok=True)
joblib.dump(clf, model_dir / 'model.joblib')
joblib.dump(scaler, model_dir / 'scaler.joblib')
print(f"Model artifacts saved to {model_dir}")

Test accuracy: 100.00%
Model artifacts saved to /tmp/vertex_model


## 4. Upload the Model Artifact to GCS

Vertex AI pulls model files from Google Cloud Storage (GCS). You must upload your artifact there before registering a Model.

In [3]:
# WHAT: upload the artifacts to a GCS bucket — or print the exact commands if no
# GCP credentials are available.
# WHY: like S3 for SageMaker, GCS is the handoff point: Vertex AI deploys models
# FROM buckets, never from your laptop directly.
GCS_BUCKET = 'my-vertex-models-bucket'
GCS_PREFIX = 'iris-classifier/v1'
GCS_MODEL_URI = f'gs://{GCS_BUCKET}/{GCS_PREFIX}'

try:
    from google.cloud import storage
    from google.auth.exceptions import DefaultCredentialsError

    gcs_client = storage.Client()
    bucket = gcs_client.bucket(GCS_BUCKET)

    for artifact in model_dir.iterdir():
        blob = bucket.blob(f'{GCS_PREFIX}/{artifact.name}')
        blob.upload_from_filename(str(artifact))
        print(f"Uploaded: gs://{GCS_BUCKET}/{GCS_PREFIX}/{artifact.name}")

# Teaching fallback: show the production upload code instead of failing the class.
except Exception as e:
    print(f"[No credentials] {type(e).__name__}")
    print(f"Would upload to: {GCS_MODEL_URI}/")
    print()
    print("In production:")
    print("  from google.cloud import storage")
    print("  client = storage.Client()")
    print("  bucket = client.bucket('my-vertex-models-bucket')")
    print("  blob = bucket.blob('iris-classifier/v1/model.joblib')")
    print("  blob.upload_from_filename('/tmp/vertex_model/model.joblib')")

[No credentials] OSError
Would upload to: gs://my-vertex-models-bucket/iris-classifier/v1/

In production:
  from google.cloud import storage
  client = storage.Client()
  bucket = client.bucket('my-vertex-models-bucket')
  blob = bucket.blob('iris-classifier/v1/model.joblib')
  blob.upload_from_filename('/tmp/vertex_model/model.joblib')


## 5. Initialize Vertex AI and Register the Model

In [4]:
# WHAT: register the GCS artifact as a Vertex AI Model with a pre-built sklearn
# serving container.
# WHY: Model.upload pairs YOUR weights with Google's serving image — the same
# 'model = artifact + container' idea as SageMaker's create_model.
PROJECT_ID = 'my-gcp-project'
REGION = 'us-central1'

try:
    from google.cloud import aiplatform
    from google.auth.exceptions import DefaultCredentialsError

    # init() sets the project and region for all subsequent calls
    aiplatform.init(project=PROJECT_ID, location=REGION)
    print(f"Vertex AI initialised. Project: {PROJECT_ID}, Region: {REGION}")

    # Register the model — Vertex AI will use a pre-built sklearn container
    SKLEARN_CONTAINER = 'us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-2:latest'

    model = aiplatform.Model.upload(
        display_name='iris-classifier-v1',
        artifact_uri=GCS_MODEL_URI,
        serving_container_image_uri=SKLEARN_CONTAINER,
        description='Random Forest iris classifier',
    )
    print(f"Model registered: {model.resource_name}")

# Without credentials, print the equivalent production snippet.
except Exception as e:
    model = None
    print(f"[No credentials] {type(e).__name__}")
    print()
    print("In production:")
    print("  from google.cloud import aiplatform")
    print(f"  aiplatform.init(project='{PROJECT_ID}', location='{REGION}')")
    print()
    print("  model = aiplatform.Model.upload(")
    print("      display_name='iris-classifier-v1',")
    print(f"      artifact_uri='{GCS_MODEL_URI}',")
    print("      serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-2:latest',")
    print("  )")

Vertex AI initialised. Project: my-gcp-project, Region: us-central1


[No credentials] PermissionDenied

In production:
  from google.cloud import aiplatform
  aiplatform.init(project='my-gcp-project', location='us-central1')

  model = aiplatform.Model.upload(
      display_name='iris-classifier-v1',
      artifact_uri='gs://my-vertex-models-bucket/iris-classifier/v1',
      serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-2:latest',
  )


## 6. Create an Endpoint

In [5]:
# WHAT: create the Vertex AI Endpoint — the stable, addressable prediction URL.
# WHY: endpoint and model are separate resources so multiple model versions can
# later share one endpoint with traffic splitting.
ENDPOINT_NAME = 'iris-endpoint-aiat125'

try:
    endpoint = aiplatform.Endpoint.create(
        display_name=ENDPOINT_NAME,
        description='Iris species classifier endpoint',
        labels={'course': 'aiat125'},
    )
    print(f"Endpoint created: {endpoint.resource_name}")
    print(f"Public endpoint: https://{REGION}-aiplatform.googleapis.com/v1/{endpoint.resource_name}:predict")

# Teaching fallback keeps `endpoint = None` so later cells know to simulate.
except Exception as e:
    endpoint = None
    print(f"[No credentials] {type(e).__name__}")
    print()
    print("In production:")
    print("  endpoint = aiplatform.Endpoint.create(")
    print(f"      display_name='{ENDPOINT_NAME}',")
    print("      labels={'course': 'aiat125'},")
    print("  )")

[No credentials] PermissionDenied

In production:
  endpoint = aiplatform.Endpoint.create(
      display_name='iris-endpoint-aiat125',
      labels={'course': 'aiat125'},
  )


## 7. Deploy the Model to the Endpoint

In [6]:
# WHAT: deploy the registered model onto the endpoint with autoscaling 1-5 replicas.
# WHY: this is the step that allocates billable machines; traffic_percentage=100
# routes everything here until a newer version claims a share.
try:
    if model and endpoint:
        deployed_model = model.deploy(
            endpoint=endpoint,
            deployed_model_display_name='iris-rf-v1',
            machine_type='n1-standard-2',   # 2 vCPU, 7.5 GB RAM
            min_replica_count=1,
            max_replica_count=5,
            traffic_percentage=100,
        )
        print(f"Model deployed. Endpoint ready at: {endpoint.resource_name}")
    else:
        print("[No credentials] Would call: model.deploy(endpoint=endpoint, ...)")
        raise Exception('Simulated to show the expected output')

# The printed snippet is the exact production call, including the ~10 min wait.
except Exception as e:
    if 'Simulated' not in str(e):
        print(f"[No credentials] {type(e).__name__}")
    print()
    print("In production:")
    print("  model.deploy(")
    print("      endpoint=endpoint,")
    print("      deployed_model_display_name='iris-rf-v1',")
    print("      machine_type='n1-standard-2',")
    print("      min_replica_count=1,")
    print("      max_replica_count=5,       # auto-scales up to 5 replicas")
    print("      traffic_percentage=100,    # all traffic to this model")
    print("  )")
    print()
    print("Deploy takes ~10 minutes. The endpoint reports 'DEPLOYED' when ready.")

[No credentials] Would call: model.deploy(endpoint=endpoint, ...)

In production:
  model.deploy(
      endpoint=endpoint,
      deployed_model_display_name='iris-rf-v1',
      machine_type='n1-standard-2',
      min_replica_count=1,
      max_replica_count=5,       # auto-scales up to 5 replicas
      traffic_percentage=100,    # all traffic to this model
  )

Deploy takes ~10 minutes. The endpoint reports 'DEPLOYED' when ready.


## 8. Make a Prediction

In [7]:
# WHAT: call endpoint.predict() with two flowers — or simulate the response with
# the local model.
# WHY: the response fields (predictions, deployed_model_id) are worth reading:
# they tell you WHICH deployed version answered, essential for debugging rollouts.
# Sample input: two iris flowers
instances = [
    [5.1, 3.5, 1.4, 0.2],   # expected: setosa
    [6.7, 3.1, 4.7, 1.5],   # expected: versicolor
]

try:
    if endpoint:
        prediction = endpoint.predict(instances=instances)
        print(f"Predictions: {prediction.predictions}")
    else:
        raise Exception('no endpoint')

# Simulation path: same preprocessing + model, so the answers match the cloud's.
except Exception:
    print("[No credentials] Would call: endpoint.predict(instances=[...])")
    print(f"Input instances: {instances}")
    print()
    # Show what the response looks like using local model
    features_s = scaler.transform(np.array(instances))
    preds = clf.predict(features_s)
    probs = clf.predict_proba(features_s).max(axis=1)
    print("Simulated Vertex AI response:")
    print("  prediction.predictions =", [iris.target_names[p] for p in preds])
    print("  prediction.deployed_model_id = 'iris-rf-v1-abc123'")
    print()
    print("Full response object would contain:")
    print("  - predictions: list of model outputs")
    print("  - deployed_model_id: which version served the request")
    print("  - model_resource_name: full resource path")

[No credentials] Would call: endpoint.predict(instances=[...])
Input instances: [[5.1, 3.5, 1.4, 0.2], [6.7, 3.1, 4.7, 1.5]]

Simulated Vertex AI response:
  prediction.predictions = [np.str_('setosa'), np.str_('versicolor')]
  prediction.deployed_model_id = 'iris-rf-v1-abc123'

Full response object would contain:
  - predictions: list of model outputs
  - deployed_model_id: which version served the request
  - model_resource_name: full resource path


## 9. Undeploy and Delete to Avoid Costs

In [8]:
# Clean up the cloud resources so billing stops — or, when nothing was deployed
# in this run (no credentials), say so honestly and show the production commands.
try:
    if endpoint or model:
        # Real cleanup path: only reachable when the earlier cells deployed for real
        if endpoint:
            endpoint.undeploy_all()   # removes all deployed models from the endpoint
            endpoint.delete()         # deletes the endpoint resource
            print("Endpoint undeployed and deleted.")
        if model:
            model.delete()
            print("Model resource deleted.")
    else:
        # Both handles are None because authentication failed earlier — there is
        # nothing to delete, but the cleanup sequence still needs demonstrating.
        print("[No credentials] Nothing was deployed in this run, so there is nothing to delete.")
        print()
        print("Cleanup in production:")
        print("  endpoint.undeploy_all()  # stop compute billing")
        print("  endpoint.delete()        # remove the endpoint URL")
        print("  model.delete()           # remove the model registration")
        print("  # GCS artifact stays unless you delete it manually")

except Exception as e:
    # A real cleanup attempt failed mid-way (e.g. token expired) — show why,
    # then the same reference sequence.
    print(f"[Cleanup failed] {type(e).__name__}")
    print()
    print("Cleanup in production:")
    print("  endpoint.undeploy_all()  # stop compute billing")
    print("  endpoint.delete()        # remove the endpoint URL")
    print("  model.delete()           # remove the model registration")
    print("  # GCS artifact stays unless you delete it manually")


[No credentials] Nothing was deployed in this run, so there is nothing to delete.

Cleanup in production:
  endpoint.undeploy_all()  # stop compute billing
  endpoint.delete()        # remove the endpoint URL
  model.delete()           # remove the model registration
  # GCS artifact stays unless you delete it manually


## 💬 Discuss

The simulated prediction response carried three things: the predictions, `deployed_model_id`, and the full model resource name. That second field is the one worth arguing about.

1. Why would a prediction response include the ID of the model that produced it? Describe an incident where you would give a lot to have that field in your logs — and one where logging it creates a problem instead.
2. Vertex AI bills per second, SageMaker per hour. For a model deployed to a Saudi client whose traffic arrives in two office-hours peaks, does per-second billing actually change your architecture, or just your invoice? Show your reasoning.
3. The Google Cloud outage above spread globally because the *data* was global even though the *code* rollout was regional. Look at your own deployment: what in it is replicated globally and could arrive faster than your rollout gates? (Hint: the model artifact in GCS is one candidate. So is a feature file — see Unit 1, notebook 01.)

## 10. Vertex AI vs SageMaker Comparison

| Feature | Vertex AI (GCP) | SageMaker (AWS) |
|---|---|---|
| Model artifact storage | GCS (`gs://`) | S3 (`s3://`) |
| SDK | `google-cloud-aiplatform` | `boto3` + `sagemaker` |
| Endpoint update | `model.deploy()` to existing endpoint | `update_endpoint()` with new config |
| Training jobs | Vertex AI Training | SageMaker Training Jobs |
| Pipeline orchestration | Vertex AI Pipelines (Kubeflow-based) | SageMaker Pipelines |
| Feature store | Vertex AI Feature Store | SageMaker Feature Store |
| Notebook environment | Vertex AI Workbench | SageMaker Studio |
| Free tier | $300 GCP credit for new accounts | 2 months free tier for new accounts |
| Pricing model | Per-second billing | Per-hour billing |

Both platforms are functionally similar. The main reason to choose one over the other is usually which cloud your organization already uses.

## Summary

The Vertex AI deployment flow:
1. **Train** model locally with sklearn (or any framework)
2. **Save** artifact to local disk
3. **Upload to GCS** — Vertex AI cannot access local files directly
4. **Register** — `aiplatform.Model.upload()` pointing at the GCS URI
5. **Create Endpoint** — `aiplatform.Endpoint.create()` for the stable URL
6. **Deploy** — `model.deploy(endpoint=endpoint, machine_type=...)` to allocate compute
7. **Predict** — `endpoint.predict(instances=[...])` for inference
8. **Clean up** — `endpoint.undeploy_all()` then `endpoint.delete()` to stop billing

Key distinction: the **Model** is just a registry entry pointing to GCS. The **Endpoint** is the live serving infrastructure. A model must be *deployed* to an endpoint before it can receive traffic.

## Self-Check

1. **What is a Vertex AI 'endpoint' vs a 'model'?**
   *(Which one receives HTTP traffic? Which one is just a registry entry?)*

2. **What is GCS and why does Vertex AI need your model uploaded there?**
   *(Why can't Vertex AI just pull from your local disk?)*

3. **Name one thing Vertex AI has that is different from SageMaker.**
   *(Use the comparison table above if needed.)*

## ⚠️ Where this breaks

- **Managed means someone else's outage becomes your outage.** Nothing in your code, your model or your configuration would have prevented the 12 June 2025 event. The mitigations available to you are architectural and expensive: multiple regions, multiple providers, or a degraded-mode path that serves cached or default predictions when the endpoint is unreachable. Decide in advance which of those your service actually needs.
- **A staged rollout gates code, not data.** Google's failing code path shipped safely region by region and then failed everywhere at once when the triggering data appeared. If your deployment reads configuration, feature lists or model artifacts from a globally replicated store, that store is an un-gated path into production. Version and stage it like code.
- **Deploy latency is a real operational cost.** `model.deploy()` takes about ten minutes. That is your minimum time-to-recover for anything that requires a redeploy, which is why traffic splitting and rollback pointers matter more than deploy speed.
- **The assumption that must hold:** the pre-built serving container can load your artifact. `sklearn-cpu.1-2` means scikit-learn 1.2. An artifact pickled under a different version may fail on load or, worse, load into a subtly different object. Check the container's version against the version recorded in your model card.
- **Undeploy, then delete — in that order.** `endpoint.delete()` without `undeploy_all()` leaves compute running. And the GCS artifact stays and keeps costing storage until you remove it explicitly; cleanup is three separate deletions, not one.
- **The cheaper alternative.** Vertex AI, SageMaker and Azure ML are functionally close enough that the honest decision rule is the one in section 10: pick the cloud your organisation already uses. If you are truly cloud-agnostic, a container plus Kubernetes (Unit 4) keeps the inference code free of any provider SDK — which is the only form of portability that survives a migration.

## 📚 References

1. Baylor, D., Breck, E., Cheng, H.-T., et al. (2017). *TFX: A TensorFlow-Based Production-Scale Machine Learning Platform*. KDD. <https://research.google/pubs/tfx-a-tensorflow-based-production-scale-machine-learning-platform/>
2. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
3. Kreuzberger, D., Kühl, N., & Hirschl, S. (2022). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access. <https://arxiv.org/abs/2205.02302>
